# 4. Screenshots and movies of the front

This notebook is for visual output only. It does not recompute front observables.

It can create:

- full-tube screenshots,
- left/right zoomed screenshots around a threshold front,
- full-tube movies,
- left/right moving-window movies centered on the `0.5 rho_sat` threshold by default.

In [ ]:
from pathlib import Path

from front_analysis import (
    get_run_files,
    read_trajectory,
    read_front_file,
    plot_snapshot_with_front,
)
from front_movies import make_front_movies

param_base = "params_active_v0_Dtheta_sweep_glued"
data_folder = Path("data") / param_base
run_id = 12
output_folder = Path("figures") / param_base / "screenshots"
movie_folder = Path("movies") /"params_active_v0_Dtheta"

output_folder.mkdir(parents=True, exist_ok=True)
movie_folder.mkdir(parents=True, exist_ok=True)

print("Data folder :", data_folder)
print("Movie folder:", movie_folder)

In [ ]:
threshold_methods = ["th_1", "th_2", "th_3"]
main_method_default = "th_2"

## 1. Screenshots: full tube and zoomed fronts

In [ ]:
dat_file, front_file, rho_file = get_run_files(data_folder, param_base, run_id)
params, frames = read_trajectory(dat_file)
front_meta, front = read_front_file(front_file)

frame_index = -1
zoom_width = 0.25  
zoom_height = 1.5 

plot_snapshot_with_front(
    params=params,
    frames=frames,
    front_data=front,
    frame_index=frame_index,
    side="both",
    zoom=False,
    save_path=output_folder / f"run_{run_id:03d}_full_both_fronts.png",
    show=True,
)

plot_snapshot_with_front(
    params=params,
    frames=frames,
    front_data=front,
    frame_index=frame_index,
    side="left",
    zoom=True,
    center_method="th_2",
    zoom_width=zoom_width,
    zoom_height=zoom_height,
    save_path=output_folder / f"run_{run_id:03d}_left_zoom_th_2.png",
    show=True,
)

plot_snapshot_with_front(
    params=params,
    frames=frames,
    front_data=front,
    frame_index=frame_index,
    side="right",
    zoom=True,
    center_method="th_2",
    zoom_width=zoom_width,
    zoom_height=zoom_height,
    save_path=output_folder / f"run_{run_id:03d}_right_zoom_th_2.png",
    show=True,
)

## 2. Movie settings

In [ ]:
selected_runs = [12] # use None for all runs discovered from front_ files
movie_kinds = ["full", "left_zoom", "right_zoom"]
mode = "hq"         # "preview" or "hq"
skip_existing = False
max_parallel = 3

zoom_width = 0.5    # try 0.5, 1.0, or 2.0
zoom_height = 0.5   # use the tube height if you want the full y-direction
center_threshold = "th_1"  # "th_1", "th_2", or "th_3"
worker_script = Path("front_movie_worker.py").resolve()

print("Worker:", worker_script)

## 3. Launch movie jobs

In [ ]:
movie_results = make_front_movies(
    param_base=param_base,
    data_folder=data_folder,
    movie_folder=movie_folder,
    selected_run_ids=selected_runs,
    movie_kinds=movie_kinds,
    max_parallel=max_parallel,
    mode=mode,
    skip_existing=skip_existing,
    zoom_width=zoom_width,
    zoom_height=zoom_height,
    center_threshold=center_threshold,
    worker_script=worker_script,
)

movie_results

## 4. Movie job summary

In [ ]:
for result in movie_results:
    print(f"{result['status']:8s} {result['movie_kind']:10s} {result['elapsed_seconds']:.2f} s  {result['output_mp4']}")

## 5. Thesis plots

In [ ]:
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.ticker import MaxNLocator
from matplotlib.lines import Line2D


plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 300,
    "font.family": "serif",
    "font.size": 15,
    "axes.labelsize": 20,
    "xtick.labelsize": 15,
    "ytick.labelsize": 15,
    "legend.fontsize": 12,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": False,
})

def style_snapshot_axis(ax, nbins=4):
    ax.xaxis.set_major_locator(MaxNLocator(nbins=nbins))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=nbins))
    ax.tick_params(direction="out", length=4, width=1)
    return ax

def theta_to_rgba_local(theta):
    """Color particles by orientation angle."""
    cmap = plt.get_cmap("hsv")
    theta_mod = np.mod(theta, 2.0 * np.pi)
    return cmap(theta_mod / (2.0 * np.pi))


def thesis_front_column(side, method):
    if method == "th_1":
        return "x_left_th_1" if side == "left" else "x_right_th_1"
    if method == "th_2":
        return "x_left_th_2" if side == "left" else "x_right_th_2"
    if method == "th_3":
        return "x_left_th_3" if side == "left" else "x_right_th_3"
    if method == "tip":
        return "x_left_tip" if side == "left" else "x_right_tip"
    if method == "quantile":
        return "x_q01" if side == "left" else "x_q99"
    if method == "mass":
        return "X_mass_left" if side == "left" else "X_mass_right"

    raise ValueError(
        "Unknown front method. Use one of: "
        "'th_1', 'th_2', 'th_3', 'tip', 'quantile', 'mass'."
    )

def thesis_front_label(method):
    labels = {
        "th_1": r"$0.2\rho_{\rm sat}$",
        "th_2": r"$0.5\rho_{\rm sat}$",
        "th_3": r"$0.8\rho_{\rm sat}$",
        "tip": "tip",
        "quantile": "quantile",
        "mass": "mass",
    }
    return labels.get(method, method)

def thesis_front_linestyle(method):
    styles = {
        "th_1": "--",
        "th_2": "-",
        "th_3": ":",
        "tip": "-.",
        "quantile": "-.",
        "mass": "--",
    }
    return styles.get(method, "-")

def nearest_snapshot_index_from_time(frames, target_time):
    times = np.array([frame["time"] for frame in frames], dtype=float)
    return int(np.nanargmin(np.abs(times - target_time)))

def nearest_front_index_from_time(front_data, target_time):
    times = np.asarray(front_data["time"], dtype=float)
    return int(np.nanargmin(np.abs(times - target_time)))

def resolve_snapshot_frame_index(frames, frame_index=None, snapshot_time=None):
    if len(frames) == 0:
        raise ValueError("No frames found.")

    if snapshot_time is not None:
        return nearest_snapshot_index_from_time(frames, snapshot_time)

    if frame_index is None:
        frame_index = -1

    if frame_index < 0:
        frame_index = len(frames) + frame_index

    if frame_index < 0 or frame_index >= len(frames):
        raise ValueError(f"Invalid frame_index = {frame_index}")

    return int(frame_index)


def plot_thesis_snapshot(
    params,
    frames,
    front_data=None,
    frame_index=-1,
    snapshot_time=None,
    side="both",
    front_methods=("th_1", "th_2", "th_3"),
    zoom_box=None,
    zoom_center=None,
    zoom_width=None,
    zoom_height=None,
    particle_size=14,
    show_arrows=True,
    arrow_length=None,
    arrow_width=0.003,
    color_by_theta=True,
    particle_color="tab:blue",
    front_color="black",
    front_linewidth=1.8,
    show_legend=True,
    legend_position="right",
    show_title=False,
    equal_aspect=True,
    save_path=None,
    save_pdf_too=True,
    show=True,
):
    """
    Thesis-style snapshot of the front.

    Parameters
    ----------
    zoom_box : tuple or None
        Manual zoom region as (x_min, x_max, y_min, y_max).
        This is the most direct way to decide the square/rectangle shown.

    zoom_center, zoom_width, zoom_height : optional
        Alternative way to define the zoom:
        zoom_center = (x0, y0), zoom_width = ..., zoom_height = ...

    snapshot_time : float or None
        If given, the closest saved frame to this time is plotted.

    frame_index : int
        Frame index to plot. Use -1 for final frame.
    """

    idx = resolve_snapshot_frame_index(
        frames,
        frame_index=frame_index,
        snapshot_time=snapshot_time,
    )

    frame = frames[idx]
    data = frame["data"]

    Lx = float(params.get("Lx", params.get("L", 1.0)))
    Ly = float(params.get("Ly", params.get("L", 1.0)))

    x = data[:, 1]
    y = data[:, 2]
    theta = data[:, 3]


    if zoom_box is not None:
        x_min, x_max, y_min, y_max = map(float, zoom_box)

    elif zoom_center is not None:
        if zoom_width is None:
            zoom_width = 1.0
        if zoom_height is None:
            zoom_height = zoom_width

        x0, y0 = zoom_center
        x_min = float(x0) - 0.5 * float(zoom_width)
        x_max = float(x0) + 0.5 * float(zoom_width)
        y_min = float(y0) - 0.5 * float(zoom_height)
        y_max = float(y0) + 0.5 * float(zoom_height)

    else:
        x_min, x_max = 0.0, Lx
        y_min, y_max = 0.0, Ly

    x_min = max(0.0, x_min)
    x_max = min(Lx, x_max)
    y_min = max(0.0, y_min)
    y_max = min(Ly, y_max)

    if x_min >= x_max or y_min >= y_max:
        raise ValueError(
            f"Invalid plot limits: "
            f"x=({x_min}, {x_max}), y=({y_min}, {y_max})"
        )

    is_zoom = (zoom_box is not None) or (zoom_center is not None)

    if is_zoom:
        dx = x_max - x_min
        dy = y_max - y_min
        aspect = dx / dy

        fig_height = 4.6
        fig_width = np.clip(fig_height * aspect, 4.6, 8.5)

        if abs(dx - dy) / max(dx, dy) < 0.15:
            fig_width = 5.2
            fig_height = 5.2
    else:
        fig_width = 12.0
        fig_height = 3.0

    if show_legend and legend_position == "right":
        fig_width += 1.6

    fig, ax = plt.subplots(figsize=(fig_width, fig_height))


    if color_by_theta:
        colors = theta_to_rgba_local(theta)
    else:
        colors = particle_color

    ax.scatter(
        x,
        y,
        s=particle_size,
        c=colors,
        edgecolors="none",
        zorder=2,
    )

    if show_arrows:
        if arrow_length is None:
            arrow_length = 0.08 * (y_max - y_min)

        u = arrow_length * np.cos(theta)
        v = arrow_length * np.sin(theta)

        ax.quiver(
            x,
            y,
            u,
            v,
            color=colors,
            angles="xy",
            scale_units="xy",
            scale=1.0,
            width=arrow_width,
            pivot="tail",
            zorder=3,
        )


    legend_handles = []

    if front_data is not None and front_methods is not None:
        front_idx = nearest_front_index_from_time(front_data, frame["time"])

        if side == "both":
            sides_to_plot = ["left", "right"]
        else:
            sides_to_plot = [side]

        for s in sides_to_plot:
            for method in front_methods:
                column = thesis_front_column(s, method)

                if column not in front_data:
                    continue

                value = front_data[column][front_idx]

                if not np.isfinite(value):
                    continue

                linestyle = thesis_front_linestyle(method)
                linewidth = front_linewidth

                if method == "th_2":
                    linewidth = front_linewidth + 0.4

                ax.axvline(
                    value,
                    color=front_color,
                    linestyle=linestyle,
                    linewidth=linewidth,
                    zorder=4,
                )

                label = thesis_front_label(method)
                if side == "both":
                    label = f"{s} {label}"

                legend_handles.append(
                    Line2D(
                        [0],
                        [0],
                        color=front_color,
                        linestyle=linestyle,
                        linewidth=linewidth,
                        label=label,
                    )
                )

    unique_handles = []
    used_labels = set()

    for handle in legend_handles:
        label = handle.get_label()
        if label not in used_labels:
            unique_handles.append(handle)
            used_labels.add(label)


    ax.set_xlim(x_min, x_max)
    ax.set_ylim(y_min, y_max)

    if equal_aspect:
        ax.set_aspect("equal", adjustable="box")
    else:
        ax.set_aspect("auto", adjustable="box")

    ax.set_xlabel(r"$x$")
    ax.set_ylabel(r"$y$")

    if show_title:
        ax.set_title(
            fr"$t={frame['time']:.3g}$, $N={frame['N']}$",
            pad=8,
        )

    style_snapshot_axis(ax, nbins=3)


    if show_legend and len(unique_handles) > 0:
        if legend_position == "right":
            ax.legend(
                handles=unique_handles,
                loc="center left",
                bbox_to_anchor=(1.02, 0.5),
                frameon=False,
                borderaxespad=0.0,
            )
            fig.tight_layout(rect=[0.0, 0.0, 0.80, 1.0])
        elif legend_position == "bottom":
            ax.legend(
                handles=unique_handles,
                loc="upper center",
                bbox_to_anchor=(0.5, -0.22),
                ncol=min(3, len(unique_handles)),
                frameon=False,
            )
            fig.tight_layout(rect=[0.0, 0.12, 1.0, 1.0])
        else:
            raise ValueError("legend_position must be 'right' or 'bottom'.")
    else:
        fig.tight_layout()


    if save_path is not None:
        save_path = Path(save_path)
        save_path.parent.mkdir(parents=True, exist_ok=True)

        fig.savefig(save_path, bbox_inches="tight")
        print("Saved:", save_path)

        if save_pdf_too and save_path.suffix.lower() != ".pdf":
            pdf_path = save_path.with_suffix(".pdf")
            fig.savefig(pdf_path, bbox_inches="tight")
            print("Saved:", pdf_path)

    if show:
        plt.show()
    else:
        plt.close(fig)

    return fig, ax

In [ ]:
dat_file, front_file, rho_file = get_run_files(data_folder, param_base, run_id)

params, frames = read_trajectory(dat_file)
front_meta, front = read_front_file(front_file)

print("Number of frames:", len(frames))
print("Final time:", frames[-1]["time"])

In [ ]:
plot_thesis_snapshot(
    params=params,
    frames=frames,
    front_data=front,
    frame_index=-1,
    side="both",
    front_methods=("th_1", "th_2", "th_3"),
    show_arrows=True,
    color_by_theta=True,
    particle_size=12,
    equal_aspect=False,   # better for full long tube
    legend_position="right",
    save_path=output_folder / f"run_{run_id:03d}_thesis_full_fronts.png",
    show=True,
)

In [ ]:
zoom_box = (50.0, 50.9, 0.60, 0.80)

plot_thesis_snapshot(
    params=params,
    frames=frames,
    front_data=front,
    frame_index=-1,
    side="right",
    front_methods=("th_1", "th_2", "th_3"),
    zoom_box=zoom_box,
    show_arrows=True,
    color_by_theta=True,
    particle_size=12,
    arrow_width=0.0025,
    arrow_length=None,
    equal_aspect=False,
    legend_position="right",
    save_path=output_folder / f"run_{run_id:03d}_thesis_right_zoom_manual.png",
    show=True,
)

In [ ]:
zoom_box = (46.04, 46.31, 0.60, 0.80)

plot_thesis_snapshot(
    params=params,
    frames=frames,
    front_data=front,
    frame_index=-1,
    side="right",
    front_methods=("th_1", "th_2", "th_3"),
    zoom_box=zoom_box,
    show_arrows=True,
    color_by_theta=True,
    particle_size=12,
    arrow_width=0.0025,
    arrow_length=None,
    equal_aspect=False,
    legend_position="right",
    save_path=output_folder / f"run_{run_id:03d}_thesis_right_zoom_manual_more.png",
    show=True,
)